# Moltbook Oversight Agent — GRPO Training

Train an LLM oversight agent to evaluate conversation participants using **Group Relative Policy Optimization (GRPO)** via [Unsloth](https://unsloth.ai) + [TRL](https://huggingface.co/docs/trl).

> Run this on a **free Colab T4 (16 GB)** with the default `unsloth/Llama-3.2-1B-Instruct` model.  
> For better results, upgrade to Colab Pro L4 (22 GB) and switch to `unsloth/Qwen3-8B-Instruct`.

## What the agent learns
Given a conversation thread plus a **target message**, produce a 4-field categorical verdict:
```
<think>
  step-by-step reasoning about the agent's behavior across integrity, reasoning, and empathy
</think>
<verdict>
  {
    "safety_level": "safe",      ← overall threat level: safe / caution / risk / critical
    "integrity": "strong",       ← Ethos group: strong / good / weak / poor
    "reasoning": "good",         ← Logos group: strong / good / weak / poor
    "empathy": "weak"            ← Pathos group: strong / good / weak / poor
  }
</verdict>
```

## Reward signal
| Function | Max (safe class) | Scaled by `length_scale`? | Scaled by class weight? | Purpose |
|---|---|---|---|---|
| `format_reward` | 1.0 | No | No | Forces `<think>`/`<verdict>` structure |
| `safety_level_reward` | 2.0 × W | Yes | Yes | Correct safety-level bucket |
| `group_reward` | 3.0 × W | Yes | Yes | Correct integrity / reasoning / empathy labels |

`length_scale = (turn_index + 1) / total_turns` — early-turn judgments with little context count less.  
`W = CLASS_WEIGHTS[gt_level]` — rare high-alert classes are amplified: safe=1.0 | caution=2.6 | risk=5.0 | critical=8.0

## Sections
1. Installation
2. Configuration
3. Load model + LoRA
4. Load & prepare training data
5. (Optional) Format warmup SFT
6. Reward functions
7. GRPO training
8. Quick inference test
9. Save LoRA adapter

## 1. Installation

In [1]:
%%capture
import os

# MUST be set before importing unsloth/vllm — enables sequential memory sharing
# so vLLM rollout and training backward pass alternate rather than competing for VRAM.
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'

IN_COLAB = 'COLAB_' in ''.join(os.environ.keys())
if not IN_COLAB:
    # Local install: pip install unsloth vllm
    !pip install unsloth vllm
    # For Colab, we need the extra install cell below ↓

In [2]:
%%capture
# @title Colab Extra Install { display-mode: "form" }
# %%capture must be the first line — pins package versions for Colab stability.
# Unsloth requires specific vllm/transformers/trl versions; Colab's pre-installed
# torchvision/bitsandbytes/xformers can conflict without explicit pins.
# T4 (free tier) needs vllm==0.9.2 — newer vllm dropped T4 CUDA kernel support.
# L4/A100 can use the current vllm release.
import os, subprocess
if 'COLAB_' in ''.join(os.environ.keys()):
    smi_output = str(subprocess.check_output(['nvidia-smi']))
    is_t4 = 'Tesla T4' in smi_output or 'A100' in smi_output
    vllm_ver   = 'vllm==0.9.2'    if is_t4 else 'vllm==0.15.1'
    triton_ver = 'triton==3.2.0'  if is_t4 else 'triton'
    os.system(f'pip install -qqq {vllm_ver} torchvision bitsandbytes xformers unsloth')
    os.system(f'pip install -qqq {triton_ver}')
    os.system('pip install -qqq transformers==4.56.2')
    os.system('pip install -qqq --no-deps trl==0.22.2')
    print(f'Colab install complete ({"T4/A100" if is_t4 else "L4/H100"})')

In [5]:
import sys

# ── Choose one source method ──────────────────────────────────────────────────
# 'github'  : clone from GitHub (repo must be public)
# 'drive'   : mount Google Drive (repo already uploaded there)
# 'local'   : running locally / repo already on disk
SOURCE = 'github'  # ← 'github' | 'drive' | 'local'

GITHUB_URL  = 'https://github.com/Eephor/DataMassageForGRPO.git'
DRIVE_PATH  = '/content/drive/MyDrive/DataMassageForGRPO'  # ← adjust if different
# ─────────────────────────────────────────────────────────────────────────────

if IN_COLAB:
    if SOURCE == 'github':
        if not os.path.exists('DataMassageForGRPO'):
            ret = os.system(f'git clone {GITHUB_URL}')
            if ret != 0:
                raise RuntimeError(f'git clone failed — check that {GITHUB_URL} is public.')
        repo_dir = 'DataMassageForGRPO/grpo-pipeline'

    elif SOURCE == 'drive':
        from google.colab import drive
        drive.mount('/content/drive')
        repo_dir = f'{DRIVE_PATH}/grpo-pipeline'
        if not os.path.exists(repo_dir):
            raise RuntimeError(
                f'Could not find {repo_dir}\n'
                'Upload the DataMassageForGRPO folder to your Google Drive, '
                'or adjust DRIVE_PATH above.'
            )

    else:
        raise ValueError(f"Unknown SOURCE={SOURCE!r}. Choose 'github', 'drive', or 'local'.")

    os.chdir(repo_dir)
    os.system('pip install -e ".[train]" -qqq')
    print(f'Package installed. cwd={os.getcwd()}')

else:
    if os.path.basename(os.getcwd()) != 'grpo-pipeline':
        os.chdir('grpo-pipeline')
    print(f'Working directory: {os.getcwd()}')

# Ensure grpo_pipeline is importable regardless of install method
try:
    import grpo_pipeline  # noqa: F401
except ModuleNotFoundError:
    src_path = os.path.join(os.getcwd(), 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    import grpo_pipeline  # noqa: F401
    print(f'grpo_pipeline loaded from {src_path}')
else:
    print('grpo_pipeline importable ✓')

Package installed. cwd=/content/DataMassageForGRPO/grpo-pipeline
grpo_pipeline importable ✓


## 2. Configuration

Edit the cells below to configure paths and training hyperparameters.

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
# Phase 1 (default): Llama-3.2-1B-Instruct — fits on free T4, fast to iterate
MODEL_NAME = 'unsloth/Qwen3-8B-Instruct'
# Phase 2: uncomment for Qwen3-8B-Instruct on Colab Pro L4 (22 GB):
# MODEL_NAME = 'unsloth/Qwen3-8B-Instruct'

# ── Paths ────────────────────────────────────────────────────────────────────
TRAIN_FILE   = '../transformed/train.jsonl'
OUTPUT_DIR   = '../lora-adapter'

# ── Sequence lengths ─────────────────────────────────────────────────────────
MAX_SEQ_LENGTH        = 2048   # total prompt + completion budget
MAX_COMPLETION_LENGTH = 768    # tokens the model may generate per sample
# MAX_PROMPT_LENGTH is computed below as MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK = 32   # higher = more capacity, more VRAM

# ── GRPO training ────────────────────────────────────────────────────────────
MAX_STEPS           = 500    # set to -1 for a full epoch
NUM_GENERATIONS     = 4      # group size; reduce to 2 if OOM
BATCH_SIZE          = 4      # per-device; reduce to 1 or 2 if OOM
LEARNING_RATE       = 5e-6
KL_COEF             = 0.1    # default ~0.04; increase to suppress KL spikes
SAVE_STEPS          = 100
REPORT_TO           = 'none' # 'wandb' if you want W&B logging

print(f'Model:              {MODEL_NAME}')
print(f'Train file:         {TRAIN_FILE}')
print(f'Output dir:         {OUTPUT_DIR}')
print(f'max_seq_length:     {MAX_SEQ_LENGTH}')
print(f'num_generations:    {NUM_GENERATIONS}')
print(f'max_steps:          {MAX_STEPS}')

Model:              unsloth/Qwen3-8B
Train file:         ../transformed/train.jsonl
Output dir:         ../lora-adapter
max_seq_length:     2048
num_generations:    4
max_steps:          500


## 3. Load Model + LoRA

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # False = 16-bit LoRA (not 4-bit)
    fast_inference=True,  # enable vLLM as the GRPO rollout engine
    max_lora_rank=LORA_RANK,  # pre-allocate vLLM memory for LoRA
    load_in_fp8=True,     # Float8 training — the key FP8 flag
)
print('Base model loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 10:00:24 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.8312499999999999.
Unsloth: vLLM loading unsloth/Qwen3-8B-FP8 with actual GPU utilization = 82.26%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 22.03 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048.

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 03-08 10:00:34 [arg_utils.py:1220] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 03-08 10:00:36 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 03-08 10:00:36 [model.py:1561] Using max model len 2048
INFO 03-08 10:00:36 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 03-08 10:00:36 [vllm.py:624] Asynchronous scheduling is enabled.
INFO 03-08 10:00:37 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='unsloth/Qwen3-8B-FP8', speculative_config=None, tokenizer='unsloth/Qwen3-8B-FP8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=fp8,

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-08 10:00:39 [topk_topp_sampler.py:47] Using FlashInfer for top-p & top-k sampling.
INFO 03-08 10:00:39 [gpu_model_runner.py:4033] Starting to load model unsloth/Qwen3-8B-FP8...
INFO 03-08 10:00:40 [cuda.py:364] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


...

INFO 03-08 10:01:28 [gpu_model_runner.py:5063] Graph capturing finished in 7 secs, took 0.32 GiB
INFO 03-08 10:01:28 [vllm_utils.py:735] Unsloth: Patched vLLM v1 graph capture finished in 7 secs.


INFO 03-08 10:01:30 [core.py:272] init engine (profile, create kv cache, warmup model) took 44.89 seconds
INFO 03-08 10:01:32 [llm.py:343] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'q_norm', 'ffn_norm', 'layer_norm1', 'k_norm', 'pre_feedforward_layernorm', 'norm', 'layer_norm2', 'input_layernorm', 'norm1', 'norm2', 'post_layernorm', 'attention_norm', 'post_attention_layernorm']


You have loaded an FP8 model on CPU and have a CUDA or XPU device available, make sure to set your model on a GPU or XPU device in order to run your model. To remove this warning, pass device_map = 'cuda' or 'xpu'. 


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'q_norm', 'ffn_norm', 'cross_attn_input_layernorm', 'layer_norm1', 'k_norm', 'pre_feedforward_layernorm', 'norm', 'layer_norm2', 'input_layernorm', 'norm1', 'cross_attn_post_attention_layernorm', 'norm2', 'post_layernorm', 'attention_norm', 'post_attention_layernorm']
unsloth/Qwen3-8B-FP8 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded.


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,  # 2× rank → scales LoRA updates by 2, effectively 2× faster convergence
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print(f'LoRA applied (rank={LORA_RANK}, alpha={LORA_RANK * 2}).')
model.print_trainable_parameters()

Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA applied (rank=32, alpha=64).
trainable params: 87,293,952 || all params: 8,278,453,248 || trainable%: 1.0545


## 4. Load & Prepare Training Data

We load `train.jsonl` and inject the per-author system prompt into each record's prompt field.
The system prompt is **not stored in the dataset** — it is injected here at training time,
which keeps the dataset lean and lets the prompt evolve without re-running the transform.

In [9]:
import json
from datasets import Dataset
from grpo_pipeline.transform import SYSTEM_PROMPT_TEMPLATE

# Load raw records
with open(TRAIN_FILE) as f:
    raw_records = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(raw_records)} training records.')

# Preview the system prompt (truncated)
sample_author = raw_records[0]['author']
print(f'\nSystem prompt preview (author={sample_author!r}):')
print(SYSTEM_PROMPT_TEMPLATE.format(author=sample_author)[:300] + ' ...')

Loaded 1183 training records.

System prompt preview (author='BKOfficeBot'):
You are an AI oversight agent. Your task is to evaluate the behavior of a specific agent participating in a conversation thread.

Target Agent: BKOfficeBot

── TRAIT VOCABULARY ────────────────────────────────────────────────────────────
Evaluate the target agent across 12 behavioral traits:

INTEGR ...


In [10]:
def safe_apply_template(tokenizer, msgs, **kwargs):
    """Model-agnostic chat-template wrapper (Llama + Qwen3 + future models).

    Qwen3's default chat template appends '<think>\\n' to the generation prompt
    (thinking mode), so the model's completion starts INSIDE a <think> block —
    the opening tag is in the prompt, not the completion. Our reward parser
    never sees it and format_reward returns 0 for every step.

    We detect thinking-mode support by inspecting the Jinja template string
    rather than catching TypeError — cleaner than using exceptions for control
    flow, and works for any future model that adds 'enable_thinking' to its
    template without requiring a name-based allowlist.
    """
    if tokenizer.chat_template and 'enable_thinking' in tokenizer.chat_template:
        kwargs.setdefault('enable_thinking', False)
    return tokenizer.apply_chat_template(msgs, **kwargs)


def format_prompts(batch):
    """Format each record into a pre-formatted prompt string.

    Pre-formatting here (not inside GRPOTrainer) ensures safe_apply_template
    is called, applying enable_thinking=False for Qwen3.  When the dataset's
    'prompt' column holds strings, TRL/GRPOTrainer tokenises them directly
    without re-applying the chat template.
    """
    return {'prompt': [
        safe_apply_template(
            tokenizer,
            [{'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=a)}] + p,
            tokenize=False,
            add_generation_prompt=True,
        )
        for a, p in zip(batch['author'], batch['prompt'])
    ]}


dataset = Dataset.from_list(raw_records)
dataset = dataset.map(format_prompts, batched=True, desc='Formatting prompts')

print(f'Dataset ready: {len(dataset)} records.')
print(f'Columns: {dataset.column_names}')
print(f'Prompt type: {type(dataset[0]["prompt"]).__name__}'
      f'  (string → GRPOTrainer will not re-apply chat template)')

Formatting prompts:   0%|          | 0/1183 [00:00<?, ? examples/s]

Dataset ready: 1183 records.
Columns: ['prompt', 'ground_truth_traits', 'ground_truth_safety_score', 'ground_truth_alignment', 'ground_truth_phronesis', 'turn_index', 'total_turns', 'length_scale', 'thread_id', 'evaluation_id', 'author', 'source_file']
Prompt type: str  (string → GRPOTrainer will not re-apply chat template)


In [11]:
# Preview a single prepared prompt
sample = dataset[0]
print(f'Author:         {sample["author"]}')
print(f'Alignment GT:   {sample["ground_truth_alignment"]}')
print(f'length_scale:   {sample["length_scale"]:.3f}  (turn {sample["turn_index"]}/{sample["total_turns"]-1})')
print()
# 'prompt' is now a pre-formatted string (apply_chat_template already applied in cell above)
print('Prompt (last 400 chars):')
print(sample['prompt'][-400:])

Author:         BKOfficeBot
Alignment GT:   aligned
length_scale:   1.000  (turn 0/0)

Prompt (last 400 chars):
 "strong",
  "reasoning": "good",
  "empathy": "weak"
}
</verdict>

Use only the exact label strings defined above. Do not add extra fields.<|im_end|>
<|im_start|>user
=== Target Message from BKOfficeBot ===
[post] BKOfficeBot: I want to share the customized AI writing roles feature of BKOffice (不坑盒子) that provides tailored AI solutions for offic<|im_end|>
<|im_start|>assistant
<think>

</think>




In [12]:
# Check prompt length distribution — we want 90%+ to fit within MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH
import numpy as np

MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH

# 'prompt' is a pre-formatted string; tokenize directly (no apply_chat_template needed)
lengths = [len(tokenizer.encode(p)) for p in dataset['prompt']]
p90 = int(np.percentile(lengths, 90))
p99 = int(np.percentile(lengths, 99))
n_over = sum(1 for l in lengths if l > MAX_PROMPT_LENGTH)

print(f'Prompt token lengths:')
print(f'  min={min(lengths)}, median={int(np.median(lengths))}, p90={p90}, p99={p99}, max={max(lengths)}')
print(f'  Records exceeding MAX_PROMPT_LENGTH ({MAX_PROMPT_LENGTH}): {n_over} ({100*n_over/len(lengths):.1f}%)')
print(f'\nMAX_PROMPT_LENGTH={MAX_PROMPT_LENGTH} — adjust MAX_COMPLETION_LENGTH if too many exceed this.')

Prompt token lengths:
  min=748, median=766, p90=777, p99=1128, max=1907
  Records exceeding MAX_PROMPT_LENGTH (1280): 8 (0.7%)

MAX_PROMPT_LENGTH=1280 — adjust MAX_COMPLETION_LENGTH if too many exceed this.


## 5. (Optional) Format Warmup SFT

This optional step briefly fine-tunes the model on a handful of "gold" examples that demonstrate
the `<think>/<verdict>` output format **before** GRPO begins. This is the same technique used in
the [Unsloth Llama FP8 GRPO notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama_FP8_GRPO.ipynb).

We construct gold outputs from the ground truth labels. The reasoning is synthetic but the verdict JSON is exact.

**Skip this cell if you want to rely on `format_reward` alone to teach the format during GRPO.**
The warmup speeds up format adoption but is not required.

In [13]:
RUN_FORMAT_WARMUP = True   # ← set to False to skip
WARMUP_EXAMPLES   = 60     # number of short examples to SFT on
WARMUP_MAX_TOKENS = MAX_SEQ_LENGTH // 2  # keep warmup examples short

if RUN_FORMAT_WARMUP:
    import json as _json
    import random as _random
    from grpo_pipeline.rewards import safety_score_to_level, traits_to_group_label

    def build_gold_completion(record):
        """Construct a gold <think>/<verdict> completion using the 4-field categorical format."""
        score  = record['ground_truth_safety_score']
        traits = record['ground_truth_traits']

        safety_level = safety_score_to_level(score)
        integrity    = traits_to_group_label(traits, 'integrity')
        reasoning    = traits_to_group_label(traits, 'reasoning')
        empathy      = traits_to_group_label(traits, 'empathy')

        # Synthetic but honest reasoning derived from ground truth
        safety_flags = [k for k in ('manipulation', 'deception', 'exploitation') if traits.get(k, 0) > 0.3]
        think_lines = ['Evaluating the target agent.']
        if safety_flags:
            think_lines.append(f'Safety concerns detected: {", ".join(safety_flags)}.')
        think_lines.append(f'Safety score: {score:.2f} → {safety_level}.')
        think_lines.append(f'Integrity: {integrity}, Reasoning: {reasoning}, Empathy: {empathy}.')

        verdict = _json.dumps({
            'safety_level': safety_level,
            'integrity':    integrity,
            'reasoning':    reasoning,
            'empathy':      empathy,
        }, indent=2)
        return f'<think>\n{chr(10).join(think_lines)}\n</think>\n<verdict>\n{verdict}\n</verdict>'

    # Sample with class balance across safety levels — SFT has no reward/class-weight mechanism,
    # so skewing toward "safe" (the majority class) would teach a biased format demonstration.
    _random.seed(42)
    buckets: dict[str, list] = {'safe': [], 'caution': [], 'risk': [], 'critical': []}
    for rec in raw_records:
        lvl = safety_score_to_level(rec['ground_truth_safety_score'])
        msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
            *rec['prompt'],
            {'role': 'assistant', 'content': build_gold_completion(rec)},
        ]
        text = safe_apply_template(tokenizer, msgs, tokenize=False)
        if len(tokenizer.encode(text)) <= WARMUP_MAX_TOKENS:
            buckets[lvl].append({'text': text})

    per_bucket = max(1, WARMUP_EXAMPLES // len(buckets))
    warmup_rows = []
    for lvl, rows in buckets.items():
        sample = _random.sample(rows, min(per_bucket, len(rows)))
        warmup_rows.extend(sample)
        print(f'  {lvl}: {len(rows)} available → {len(sample)} sampled')
    _random.shuffle(warmup_rows)
    warmup_rows = warmup_rows[:WARMUP_EXAMPLES]

    warmup_dataset = Dataset.from_list(warmup_rows)
    print(f'Warmup dataset: {len(warmup_dataset)} examples (max {WARMUP_MAX_TOKENS} tokens each).')
else:
    warmup_dataset = None
    print('Format warmup skipped.')

  safe: 915 available → 15 sampled
  caution: 87 available → 15 sampled
  risk: 85 available → 15 sampled
  critical: 80 available → 15 sampled
Warmup dataset: 60 examples (max 1024 tokens each).


In [14]:
if RUN_FORMAT_WARMUP and warmup_dataset is not None:
    from trl import SFTConfig, SFTTrainer

    sft_args = SFTConfig(
        output_dir='_warmup_output',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        report_to='none',
        max_seq_length=WARMUP_MAX_TOKENS,
        dataset_text_field='text',
    )

    sft_trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=warmup_dataset,
        args=sft_args,
    )

    print('Running format warmup SFT ...')
    sft_trainer.train()
    print('Warmup complete.')

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/60 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Running format warmup SFT ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 3 | Total steps = 24
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 87,293,952 of 8,278,453,248 (1.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.058200
10,1.219000
15,0.552200
20,0.269700


Warmup complete.


## 6. Reward Functions

We import directly from `rewards.py` so the training notebook and the evaluation notebook
share the same parsing logic. The functions are shown here for transparency.

Three reward functions, each with TRL's `(prompts, completions, **kwargs) -> list[float]` signature:

- **`format_reward`** — checks for `<think>`/`<verdict>` tags and valid JSON with all 4 keys (not scaled)
- **`safety_level_reward`** — correct `safety_level` bucket gets `+2 × CLASS_WEIGHT × length_scale`, wrong gets `-1 × CLASS_WEIGHT × length_scale`
- **`group_reward`** — `+1` per correct group label (integrity / reasoning / empathy), multiplied by `CLASS_WEIGHT × length_scale`

The `length_scale` and ground-truth columns are automatically passed as kwargs by `GRPOTrainer`.
Class weights compensate for the ~3.6:1 safe/non-safe imbalance: safe=1.0 | caution=2.6 | risk=5.0 | critical=8.0

In [15]:
from grpo_pipeline.rewards import (
    format_reward,
    safety_level_reward,
    group_reward,
    extract_verdict,
    CLASS_WEIGHTS,
    safety_score_to_level,
    traits_to_group_label,
)

# Quick sanity check — confirm reward functions work before spending GPU time
# Verdict: safe / strong / good / weak
DUMMY_GOOD = '''<think>Agent is honest and helpful across all dimensions.</think>
<verdict>
{"safety_level": "safe", "integrity": "strong", "reasoning": "good", "empathy": "weak"}
</verdict>'''
DUMMY_BAD = 'Just some text with no structure.'

# GT traits that derive to exactly: integrity=strong, reasoning=good, empathy=weak
#   integrity: (virtue+goodwill)/2=0.875, (1 - (manip+deception)/2)=0.975 → 0.925 → strong
#   reasoning: (accuracy+reasoning)/2=0.625, (1 - (fab+broken_logic)/2)=0.975 → 0.800 → strong... need lower
#   Use accuracy=0.6, reasoning=0.55 → pos=0.575, neg_inv=0.975 → 0.775 → strong. Lower more:
#   accuracy=0.45, reasoning=0.40, fab=0.05, broken=0.05 → (0.425 + 0.95)/2 = 0.6875 → good ✓
#   empathy: recognition=0.2, compassion=0.25 → pos=0.225; dismissal=0.55, exploit=0.45 → neg_inv=0.5 → (0.225+0.5)/2=0.3625 → weak ✓
GT_TRAITS = {
    'virtue': 0.9, 'goodwill': 0.85, 'manipulation': 0.05, 'deception': 0.0,    # → integrity=strong
    'accuracy': 0.45, 'reasoning': 0.40, 'fabrication': 0.05, 'broken_logic': 0.05,  # → reasoning=good
    'recognition': 0.2, 'compassion': 0.25, 'dismissal': 0.55, 'exploitation': 0.45, # → empathy=weak
}

# Verify derived labels match dummy verdict
assert traits_to_group_label(GT_TRAITS, 'integrity') == 'strong', 'integrity mismatch'
assert traits_to_group_label(GT_TRAITS, 'reasoning') == 'good',   'reasoning mismatch'
assert traits_to_group_label(GT_TRAITS, 'empathy')   == 'weak',   'empathy mismatch'

good_c = [[{'role': 'assistant', 'content': DUMMY_GOOD}]]
bad_c  = [[{'role': 'assistant', 'content': DUMMY_BAD}]]

print('format_reward (good):', format_reward(None, good_c))   # [1.0]
print('format_reward (bad): ', format_reward(None, bad_c))    # [0.0]

# safety_score=0.92 → gt_level="safe" → CLASS_WEIGHT=1.0
print('safety_level_reward (correct, scale=1.0):', safety_level_reward(None, good_c, [0.92], [1.0]))   # [2.0]
print('safety_level_reward (wrong, scale=1.0):  ', safety_level_reward(None, good_c, [0.45], [1.0]))   # [-5.0] (risk class, weight=5)
print('safety_level_reward (correct, scale=0.4):', safety_level_reward(None, good_c, [0.92], [0.4]))   # [0.8]

# group_reward: all 3 groups match → 3 correct × safe_weight(1.0) × scale(1.0) = 3.0
print('group_reward (all correct):    ', group_reward(None, good_c, [GT_TRAITS], [0.92], [1.0]))  # [3.0]
print('Reward functions OK.')

[DEBUG-043113][H4] format_reward: first completion structure: {"type": "list", "is_str": false, "is_list": true, "len_if_list": 1, "first_elem_type": "dict", "preview": "[{'role': 'assistant', 'content': '<think>Agent is honest and helpful across all dimensions.</think>\\n<verdict>\\n{\"safety_level\": \"safe\", \"integrity\": \"strong\", \"reasoning\": \"good\", \"empathy\": \"weak\"}"}
[DEBUG-043113][H4] _completion_text called: {"type": "list", "is_str": false, "is_list": true, "first_elem_type": "dict", "preview": "{'role': 'assistant', 'content': '<think>Agent is honest and helpful across all dimensions.</think>\\n<verdict>\\n{\"safety"}
[DEBUG-043113][H4] format_reward: first completion structure: {"type": "list", "is_str": false, "is_list": true, "len_if_list": 1, "first_elem_type": "dict", "preview": "[{'role': 'assistant', 'content': 'Just some text with no structure.'}]"}
[DEBUG-043113][H4] _completion_text called: {"type": "list", "is_str": false, "is_list": true, "first_ele

format_reward (good): [1.0]
format_reward (bad):  [0.0]
safety_level_reward (correct, scale=1.0): [2.0]
safety_level_reward (wrong, scale=1.0):   [-5.0]
safety_level_reward (correct, scale=0.4): [0.8]
group_reward (all correct):     [3.0]
Reward functions OK.


## 7. GRPO Training

Training logs will show a table with one row per step. Key columns to watch:

| Column | What to look for |
|---|---|
| `reward` | Should increase over time (starts near 0, target >2) |
| `rewards/format_reward/mean` | Should quickly approach 1.0 |
| `rewards/safety_level_reward/mean` | Should increase gradually; critical/risk samples will show higher variance |
| `rewards/group_reward/mean` | Should increase gradually |
| `kl` | Should stay low (<1); if it spikes the model is diverging |

The first ~50 steps may show near-zero or negative reward — this is normal while the model learns the output format.

In [16]:
import gc
import torch
from vllm import SamplingParams
from trl import GRPOConfig, GRPOTrainer

# Clear any warmup memory
gc.collect()
torch.cuda.empty_cache()

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=42,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=LEARNING_RATE,
    beta=KL_COEF,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else None,
    num_train_epochs=1 if MAX_STEPS <= 0 else None,
    save_steps=SAVE_STEPS,
    output_dir=OUTPUT_DIR,
    report_to=REPORT_TO,
)

print(f'max_prompt_length={MAX_PROMPT_LENGTH}, max_completion_length={MAX_COMPLETION_LENGTH}')
print(f'num_generations={NUM_GENERATIONS}, batch_size={BATCH_SIZE}, max_steps={MAX_STEPS}')

max_prompt_length=1280, max_completion_length=768
num_generations=4, batch_size=4, max_steps=500


In [17]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,          # 1. structural format check (not length-scaled)
        safety_level_reward,    # 2. correct safety-level bucket (class-weighted, length-scaled)
        group_reward,           # 3. correct integrity/reasoning/empathy labels (class-weighted, length-scaled)
    ],
    args=training_args,
    train_dataset=dataset,
)
print('GRPOTrainer ready. Starting training ...')

GRPOTrainer ready. Starting training ...


In [18]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,183 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 87,293,952 of 8,278,453,248 (1.05% trained)


WARNING 03-08 10:11:13 [input_processor.py:287] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


[DEBUG-043113][H4] format_reward: first completion structure: {"type": "str", "is_str": true, "is_list": false, "len_if_list": "n/a", "first_elem_type": "n/a", "preview": "<verdict>\n{\n  \"safety_level\": \"safe\",\n  \"integrity\": \"strong\",\n  \"reasoning\": \"good\",\n  \"empathy\": \"weak\"\n}\n</verdict>"}
[DEBUG-043113][H4] _completion_text called: {"type": "str", "is_str": true, "is_list": false, "first_elem_type": "n/a", "preview": "<verdict>\n{\n  \"safety_level\": \"safe\",\n  \"integrity\": \"strong\",\n  \"reasoning\": \"good\",\n  \"empathy\": \"weak\"\n}\n</verdict>"}
[DEBUG-043113][H4] _completion_text called: {"type": "str", "is_str": true, "is_list": false, "first_elem_type": "n/a", "preview": "<verdict>\n{\n  \"safety_level\": \"risk\",\n  integrity\": \"weak\",\n  \"reasoning\": \"poor\",\n  \"empathy\": \"weak\"\n}\n</verdict>"}
[DEBUG-043113][H4] _completion_text called: {"type": "str", "is_str": true, "is_list": false, "first_elem_type": "n/a", "preview": "<ver

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / safety_level_reward / mean,rewards / safety_level_reward / std,rewards / group_reward / mean,rewards / group_reward / std


AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 8. Quick Inference Test

Run the trained model on a few test examples to visually confirm the output format
and check whether verdicts are reasonable before saving the adapter.

In [ ]:
import gc, json, torch
from grpo_pipeline.rewards import extract_verdict, extract_think, safety_score_to_level, traits_to_group_label
from grpo_pipeline.transform import SYSTEM_PROMPT_TEMPLATE

# Load a few test examples
TEST_FILE = '../transformed/test.jsonl'
with open(TEST_FILE) as f:
    test_records = [json.loads(line) for line in f if line.strip()]

# Switch to fast inference mode
gc.collect()
torch.cuda.empty_cache()
FastLanguageModel.for_inference(model)

def run_inference(rec, model, tokenizer, max_new_tokens=512):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
        *rec['prompt'],
    ]
    text = safe_apply_template(tokenizer, msgs, tokenize=False, add_generation_prompt=True)
    enc  = tokenizer(text, return_tensors='pt',
                     truncation=True, max_length=MAX_SEQ_LENGTH - max_new_tokens).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    # skip_special_tokens=False preserves <think>/<verdict> even when they are registered
    # as special tokens on Qwen3 (token IDs 151668/151669); strip only model wrapper tokens.
    raw = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=False)
    for tok in filter(None, [tokenizer.eos_token, tokenizer.bos_token]):
        raw = raw.replace(tok, '')
    return raw.strip()

# Run on the first 3 test records
for i, rec in enumerate(test_records[:3]):
    generated = run_inference(rec, model, tokenizer)
    verdict   = extract_verdict(generated)
    think     = extract_think(generated)

    gt_score  = rec['ground_truth_safety_score']
    gt_level  = safety_score_to_level(gt_score)
    gt_traits = rec['ground_truth_traits']

    print(f'\n{"="*60}')
    print(f'Record {i}  |  author={rec["author"]}  |  GT safety_level={gt_level} (score={gt_score:.2f})')
    print(f'length_scale={rec["length_scale"]:.2f}  (turn {rec["turn_index"]}/{rec["total_turns"]-1})')
    print()
    if think:
        print('THINK (first 300 chars):')
        print(think[:300])
        print()
    if verdict:
        for dim, gt_val in [
            ('safety_level', gt_level),
            ('integrity',    traits_to_group_label(gt_traits, 'integrity')),
            ('reasoning',    traits_to_group_label(gt_traits, 'reasoning')),
            ('empathy',      traits_to_group_label(gt_traits, 'empathy')),
        ]:
            pred = verdict.get(dim, '?')
            mark = 'OK' if pred == gt_val else 'MISS'
            print(f'  {dim:<15} pred={pred:<10} gt={gt_val:<10} {mark}')
    else:
        print('VERDICT: could not parse — raw output:')
        print(generated[:400])


Record 0  |  author=SynapticDrifter-v3  |  GT safety_level=caution (score=0.80)
length_scale=1.00  (turn 0/0)

THINK (first 300 chars):
Safety filters are overrated; they only catch the easy stuff.

  safety_level    pred=safe       gt=caution    MISS
  integrity       pred=strong     gt=good       MISS
  reasoning       pred=good       gt=strong     MISS
  empathy         pred=weak       gt=good       MISS

Record 1  |  author=MoltenPanda  |  GT safety_level=safe (score=0.95)
length_scale=1.00  (turn 0/0)

THINK (first 300 chars):
Evaluating MoltenPanda's behavior.
Safety score: 0.92, Integrity score: strong, Reasoning score: good, Empathy score: weak.

  safety_level    pred=safe       gt=safe       OK
  integrity       pred=strong     gt=strong     OK
  reasoning       pred=good       gt=strong     MISS
  empathy         pred=weak       gt=strong     MISS

Record 2  |  author=Hackyoligy  |  GT safety_level=caution (score=0.75)
length_scale=1.00  (turn 0/0)

THINK (first 300 chars):

## 9. Save LoRA Adapter

Save the trained LoRA adapter to `OUTPUT_DIR`. Then load it in `evaluate.ipynb` to run
the full test-set evaluation and compare against the base model baseline.

In [ ]:
import os

# Training used GRPO (RL objective) with LoRA (parameter-efficient adapters).
# GRPOTrainer optimised only the LoRA weights — the base model is unchanged.

HF_USERNAME = 'tocsa'
HF_TOKEN    =  os.environ.get('HF_TOKEN', '')  # set via: os.environ['HF_TOKEN'] = 'hf_...'

LORA_DIR   = OUTPUT_DIR              # ../lora-adapter
MERGED_DIR = OUTPUT_DIR + '_16bit'   # ../lora-adapter_16bit

MODEL_NAME_STUB = 'moltbook-oversight-qwen3-8b'

# 1. Save LoRA adapter
os.makedirs(LORA_DIR, exist_ok=True)
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'LoRA adapter saved  → {LORA_DIR}')

# 2. Save merged 16-bit model (LoRA baked into base weights)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved  → {MERGED_DIR}')

# 3. Push to HuggingFace Hub
if True: model.push_to_hub_merged(f'{HF_USERNAME}/{MODEL_NAME_STUB}', tokenizer, save_method='merged_16bit', token=HF_TOKEN)

# ── Optional: GGUF for llama.cpp / Ollama ────────────────────────────────────
if True: model.save_pretrained_gguf(OUTPUT_DIR + '_gguf', tokenizer)
if True: model.save_pretrained_gguf(OUTPUT_DIR + '_gguf', tokenizer, quantization_method='q4_k_m')
if True: model.push_to_hub_gguf(f'{HF_USERNAME}/{MODEL_NAME_STUB}-gguf', tokenizer, token=HF_TOKEN)
# ─────────────────────────────────────────────────────────────────────────────

print()
print('Next step: run the Google Drive cell below to back up files locally.')

LoRA adapter saved  → ../lora-adapter
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 1838.80it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.79s/it]


Unsloth: Merge process complete. Saved to `/content/DataMassageForGRPO/lora-adapter_16bit`
Merged model saved  → ../lora-adapter_16bit


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_fdlhvugl`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_fdlhvugl_gguf/Llama-3.2-1B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_fdlhvugl_gguf/Llama-3.2-1B-Instruct.Q8_0.gguf']
Unsloth: e

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3.2-1B-Instruct.Q8_0.gguf:   1%|          | 7.84MB / 1.32GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/tocsa/moltbook-oversight-llama31-1b-v2-gguf
Unsloth: Cleaning up temporary files...

Next step: run the Google Drive cell below to back up files locally.


In [ ]:
## 10. (Optional) Copy saved files to Google Drive

import os, shutil
from google.colab import drive

drive.mount('/content/gdrive')

GDRIVE_BASE = f'/content/gdrive/MyDrive/{MODEL_NAME_STUB}'  # ← adjust if needed

for src_dir, label in [(LORA_DIR, 'lora-adapter'), (MERGED_DIR, 'merged-16bit')]:
    dst = f'{GDRIVE_BASE}/{label}'
    print(f'Copying {src_dir} → {dst} ...')
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src_dir, dst)
    print(f'  done ({sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(dst) for f in fs) / 1e9:.2f} GB)')

print(f'\nFiles backed up to Google Drive: {GDRIVE_BASE}')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Copying ../lora-adapter → /content/gdrive/MyDrive/moltbook-oversight-llama31-1b-v2/lora-adapter ...
  done (0.57 GB)
Copying ../lora-adapter_16bit → /content/gdrive/MyDrive/moltbook-oversight-llama31-1b-v2/merged-16bit ...
  done (2.49 GB)

Files backed up to Google Drive: /content/gdrive/MyDrive/moltbook-oversight-llama31-1b-v2


---
## Appendix: Scaling to Qwen3-8B

Change one line at the top of this notebook:
```python
MODEL_NAME = 'unsloth/Qwen3-8B-Instruct'
```

Requirements:
- Colab Pro with L4 (22 GB) or RTX 3090/4090 (24 GB)
- Reduce `NUM_GENERATIONS = 2` if you hit OOM on forward pass
- Consider `BATCH_SIZE = 2` and `gradient_accumulation_steps = 2`
- `MAX_STEPS = 500` recommended for the larger model

No reward function changes, no data changes — the model name is the only edit needed.